In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json
from jaxcmr.summarize import (
    calculate_aic_weights,
    generate_t_p_matrices,
    summarize_parameters,
    winner_comparison_matrix,
    raw_winner_comparison_matrix,
    pairwise_aic_differences,
)
from jaxcmr.helpers import find_project_root

In [ ]:
fit_tag = "rerun_best_of_1"
fit_dir = "projects/repfr/results/fits/"
target_directory = "projects/repfr/"

data_names = [
    "LohnasKahana2014",
    "Lohnas2025",
    "RepeatedRecallsGordonRanschburg2021",
    "RepeatedRecallsKahanaJacobs2000",
]

model_names = [
    "WeirdCMRNoStop",
    "NoReinstateCMRNoStop",
    "DistinctContextsCMRNoStop",
    "BasePositionalCMRNoStop",
    "FullPositionalCMRNoStop",
    "McfReinfPositionalCMRNoStop",
    "MfcReinfPositionalCMRNoStop",
    "FullReinfPositionalCMRNoStop",
    "SimpleFullReinfPositionalCMRNoStop",
    "BlendPositionalCMRNoStop",
]


model_titles = [
    "WeirdCMR",
    "NoReinstateCMR",
    "DistinctContextsCMR",
    "BasePositionalCMR",
    "FullPositionalCMR",
    "MCFReinfPositionalCMR",
    "MFCReinfPositionalCMR",
    "FullReinfPositionalCMR",
    "SimpleFullReinfPositionalCMR",
    "BlendPositionalCMR",
]



query_parameters = [
    "encoding_drift_rate",
    "start_drift_rate",
    "recall_drift_rate",
    "shared_support",
    "item_support",
    "learning_rate",
    "primacy_scale",
    "primacy_decay",
    "choice_sensitivity",
    "mfc_sensitivity",
    "first_pres_reinforcement",
    "mcf_first_pres_reinforcement",
    "mfc_first_pres_reinforcement",
]



In [4]:
run_tag = "Model_Comparison"

if not model_titles:
    model_titles = model_names.copy()

for data_name in data_names:
    print(f"### {data_name}\n")
    results = []

    for model_name, model_title in zip(model_names, model_titles):
        fit_path = os.path.join(find_project_root(), fit_dir, f"{data_name}_{model_name}_{fit_tag}.json")

        with open(fit_path) as f:
            results.append(json.load(f))
            if "subject" not in results[-1]["fits"]:
                results[-1]["fits"]["subject"] = results[-1]["subject"]
            if "allow_repeated_recalls" not in results[-1]["fixed"]:
                results[-1]["fixed"]["allow_repeated_recalls"] = False
                results[-1]["fits"]["allow_repeated_recalls"] = [False] * len(
                    results[-1]["fits"]["subject"]
                )
            results[-1]["name"] = model_title
            if "mfc_trace_sensitivity" in results[-1]["free"]:
                results[-1]["free"]["repetition_orthogonality"] = results[-1]["free"][
                    "mfc_trace_sensitivity"
                ]
                results[-1]["fits"]["repetition_orthogonality"] = results[-1]["fits"][
                    "mfc_trace_sensitivity"
                ]
                results[-1]["free"].pop("mfc_trace_sensitivity")
                results[-1]["fits"].pop("mfc_trace_sensitivity")

    summary = summarize_parameters(
        results, query_parameters, include_std=True, include_ci=True
    )

    table_path = os.path.join(
        find_project_root(), target_directory, "results", "tables", f"{data_name}_{fit_tag}_{run_tag}_parameters.md"
    )
    with open(table_path, "w") as f:
        f.write(summary)
    print("\nParameter Summary:")
    print(summary)

    df_t, df_p = generate_t_p_matrices(results)

    print("\nT-Test P-Value Matrix:")
    print(df_p.to_markdown())
    print()

    aic_weights = calculate_aic_weights(results)

    with open(
        os.path.join(
            find_project_root(), target_directory,
            "results", "tables",
            f"{data_name}_{fit_tag}_{run_tag}_aic_weights.md",
        ),
        "w",
    ) as f:
        f.write(aic_weights.to_markdown())

    print("\nAIC Weights:")
    print(aic_weights.to_markdown())
    print()

    df_comparison = winner_comparison_matrix(results)

    with open(
        os.path.join(
            find_project_root(), target_directory,
            "results", "tables",
            f"{data_name}_{fit_tag}_{run_tag}_winner_ratios.md",
        ),
        "w",
    ) as f:
        f.write(df_comparison.to_markdown().replace(" nan ", "     "))

    print("\nWinner Ratios (AICw):")
    print(df_comparison.to_markdown().replace(" nan ", "     "))
    print()

    df_comparison = raw_winner_comparison_matrix(results)

    with open(
        os.path.join(
            find_project_root(), target_directory,
            "results", "tables",
            f"{data_name}_{fit_tag}_{run_tag}_raw_winner_ratios.md",
        ),
        "w",
    ) as f:
        f.write(df_comparison.to_markdown().replace(" nan ", "     "))

    print("\nRaw Winner Ratios:")
    print(df_comparison.to_markdown().replace(" nan ", "     "))
    print()

    mean_delta, ci_halfwidth, _ = pairwise_aic_differences(results)

    delta_aic_table = mean_delta.copy().astype(object)
    for row_label in delta_aic_table.index:
        for col_label in delta_aic_table.columns:
            if row_label == col_label:
                delta_aic_table.loc[row_label, col_label] = ""
                continue
            mean_value = mean_delta.loc[row_label, col_label]
            ci = ci_halfwidth.loc[row_label, col_label]
            if mean_value != mean_value or ci != ci:
                delta_aic_table.loc[row_label, col_label] = ""
            else:
                lower = mean_value - ci
                upper = mean_value + ci
                delta_aic_table.loc[row_label, col_label] = f"{mean_value:.2f} [{lower:.2f}, {upper:.2f}]"

    print("\nPairwise ΔAIC (row − column; mean [CI]):")
    print(delta_aic_table.to_markdown())
    print()


### LohnasKahana2014


Parameter Summary:
| Parameter | Statistic | WeirdCMR | NoReinstateCMR | DistinctContextsCMR | BasePositionalCMR | FullPositionalCMR | MCFReinfPositionalCMR | MFCReinfPositionalCMR | FullReinfPositionalCMR | SimpleFullReinfPositionalCMR | BlendPositionalCMR |
|---|---|---|---|---|---|---|---|---|---|---|---|
| fitness | mean | 1534.29 +/- 145.36 | 1537.49 +/- 145.41 | 1528.48 +/- 145.80 | 1527.89 +/- 145.69 | 1527.79 +/- 146.06 | 1526.30 +/- 145.28 | 1527.14 +/- 145.59 | 1524.29 +/- 145.11 | 1527.24 +/- 145.54 | 1525.71 +/- 145.70 |
|  | std | 417.07 | 417.21 | 418.33 | 418.00 | 419.08 | 416.84 | 417.72 | 416.34 | 417.60 | 418.05 |
|  | min | 743.07 | 743.91 | 733.78 | 733.85 | 732.36 | 732.17 | 733.78 | 740.53 | 732.59 | 729.38 |
|  | max | 2604.49 | 2607.75 | 2599.90 | 2603.54 | 2605.28 | 2603.96 | 2599.28 | 2599.84 | 2604.62 | 2600.40 |
| encoding drift rate | mean | 0.75 +/- 0.04 | 0.74 +/- 0.04 | 0.80 +/- 0.03 | 0.67 +/- 0.04 | 0.69 +/- 0.05 | 0.72 +/- 0.04 